In [ ]:
import os
import random
import math
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

def find_all_wavs(root_dir, extensions=('.flac', '.wav')):
    files = []
    for ext in extensions:
        files.extend(Path(root_dir).rglob(f'*{ext}'))
    return [str(f) for f in files]

def load_audio(path, target_sr=8000):
    (wav, sr) = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:
        wav = T.Resample(sr, target_sr)(wav)
    return wav.squeeze(0)

def apply_gain_snr(signal, snr_db):
    power = signal.pow(2).mean() + 1e-08
    scale = 10 ** (snr_db / 20) / power.sqrt()
    return signal * scale

class DynamicMixingDataset(Dataset):

    def __init__(self, audio_dir, n_speakers=2, sample_rate=8000, segment_seconds=4.0, n_samples=20000, snr_min=-5.0, snr_max=5.0, speed_perturb=True, speed_factors=(0.9, 1.0, 1.1)):
        self.files = find_all_wavs(audio_dir)
        assert len(self.files) >= n_speakers, f'Need at least {n_speakers} audio files, found {len(self.files)}'
        self.n_speakers = n_speakers
        self.sample_rate = sample_rate
        self.segment_len = int(segment_seconds * sample_rate)
        self.n_samples = n_samples
        self.snr_min = snr_min
        self.snr_max = snr_max
        self.speed_perturb = speed_perturb
        self.speed_factors = speed_factors

    def __len__(self):
        return self.n_samples

    def _load_segment(self, path):
        wav = load_audio(path, self.sample_rate)
        if self.speed_perturb and random.random() < 0.5:
            factor = random.choice(self.speed_factors)
            if factor != 1.0:
                orig_len = len(wav)
                new_len = int(orig_len / factor)
                wav = F.interpolate(wav.unsqueeze(0).unsqueeze(0), size=new_len, mode='linear', align_corners=False).squeeze()
        if len(wav) >= self.segment_len:
            start = random.randint(0, len(wav) - self.segment_len)
            wav = wav[start:start + self.segment_len]
        else:
            repeats = math.ceil(self.segment_len / len(wav))
            wav = wav.repeat(repeats)[:self.segment_len]
        return wav

    def __getitem__(self, idx):
        chosen = random.sample(self.files, self.n_speakers)
        sources = []
        for path in chosen:
            wav = self._load_segment(path)
            snr = random.uniform(self.snr_min, self.snr_max)
            wav = apply_gain_snr(wav, snr)
            sources.append(wav)
        sources = torch.stack(sources, dim=0)
        mixture = sources.sum(dim=0)
        max_amp = mixture.abs().max() + 1e-08
        mixture = mixture / max_amp
        sources = sources / max_amp
        mixture = mixture.unsqueeze(0)
        sources = sources.unsqueeze(1)
        return (mixture, sources)

def build_dataloaders(train_audio_dir, librimix_dir=None, n_speakers=2, sample_rate=8000, segment_seconds=4.0, batch_size=4, n_train_samples=20000, num_workers=2):
    train_dataset = DynamicMixingDataset(audio_dir=train_audio_dir, n_speakers=n_speakers, sample_rate=sample_rate, segment_seconds=segment_seconds, n_samples=n_train_samples, speed_perturb=True)
    val_dataset = DynamicMixingDataset(audio_dir=train_audio_dir, n_speakers=n_speakers, sample_rate=sample_rate, segment_seconds=segment_seconds, n_samples=2000, speed_perturb=False)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class STFT(nn.Module):

    def __init__(self, n_fft=256, hop_length=128, win_length=256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer('window', torch.hann_window(win_length))

    def forward(self, x):
        return torch.stft(x, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.win_length, window=self.window, return_complex=True)

    def inverse(self, spec, length=None):
        return torch.istft(spec, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.win_length, window=self.window, length=length)

class IntraFrameFullBand(nn.Module):

    def __init__(self, embed_dim, n_heads, dropout=0.0):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, embed_dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(embed_dim * 2, embed_dim))

    def forward(self, x):
        (B, F, T, D) = x.shape
        x_bt = x.permute(0, 2, 1, 3).reshape(B * T, F, D)
        n = self.norm(x_bt)
        (a, _) = self.attn(n, n, n)
        x_bt = x_bt + a
        x_bt = x_bt + self.ff(x_bt)
        return x_bt.reshape(B, T, F, D).permute(0, 2, 1, 3)

class SubBandTemporal(nn.Module):

    def __init__(self, embed_dim, lstm_hidden, dropout=0.0):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.lstm = nn.LSTM(embed_dim, lstm_hidden, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(lstm_hidden * 2, embed_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        (B, F, T, D) = x.shape
        x_bf = x.reshape(B * F, T, D)
        (out, _) = self.lstm(self.norm(x_bf))
        x_bf = x_bf + self.drop(self.proj(out))
        return x_bf.reshape(B, F, T, D)

class CrossFrameAttention(nn.Module):

    def __init__(self, n_freqs, embed_dim, n_heads, dropout=0.0):
        super().__init__()
        self.flat_dim = n_freqs * embed_dim
        self.attn_dim = 128
        self.norm = nn.LayerNorm(self.flat_dim)
        self.down = nn.Linear(self.flat_dim, self.attn_dim)
        self.attn = nn.MultiheadAttention(self.attn_dim, n_heads, dropout=dropout, batch_first=True)
        self.up = nn.Linear(self.attn_dim, self.flat_dim)
        self.ff_norm = nn.LayerNorm(self.flat_dim)
        self.ff = nn.Sequential(nn.Linear(self.flat_dim, self.attn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(self.attn_dim, self.flat_dim))

    def forward(self, x):
        (B, F, T, D) = x.shape
        x_t = x.permute(0, 2, 1, 3).reshape(B, T, F * D)
        n = self.norm(x_t)
        d = self.down(n)
        (a, _) = self.attn(d, d, d)
        x_t = x_t + self.up(a)
        x_t = x_t + self.ff(self.ff_norm(x_t))
        return x_t.reshape(B, T, F, D).permute(0, 2, 1, 3)

class TFGridNetBlock(nn.Module):

    def __init__(self, n_freqs, embed_dim, n_heads, lstm_hidden, cross_heads, dropout=0.0):
        super().__init__()
        self.intra = IntraFrameFullBand(embed_dim, n_heads, dropout)
        self.sub = SubBandTemporal(embed_dim, lstm_hidden, dropout)
        self.cross = CrossFrameAttention(n_freqs, embed_dim, cross_heads, dropout)

    def forward(self, x):
        x = self.intra(x)
        x = self.sub(x)
        x = self.cross(x)
        return x

class TFGridNet(nn.Module):

    def __init__(self, n_speakers=2, sample_rate=8000, n_fft=256, hop_length=128, win_length=256, embed_dim=32, n_blocks=4, n_heads=4, lstm_hidden=128, cross_heads=4, dropout=0.1):
        super().__init__()
        self.n_speakers = n_speakers
        self.stft = STFT(n_fft, hop_length, win_length)
        n_freqs = n_fft // 2 + 1
        self.input_proj = nn.Linear(2, embed_dim)
        self.blocks = nn.ModuleList([TFGridNetBlock(n_freqs, embed_dim, n_heads, lstm_hidden, cross_heads, dropout) for _ in range(n_blocks)])
        self.output_proj = nn.Linear(embed_dim, n_speakers * 2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for (name, p) in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(p)
                    elif 'bias' in name:
                        nn.init.zeros_(p)

    def forward(self, mixture):
        if mixture.dim() == 3:
            mixture = mixture.squeeze(1)
        (B, T) = mixture.shape
        spec = self.stft(mixture)
        F_dim = spec.shape[1]
        T_f = spec.shape[2]
        spec_real = spec.real.clamp(-100, 100)
        spec_imag = spec.imag.clamp(-100, 100)
        x = torch.stack([spec_real, spec_imag], dim=-1)
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        out = self.output_proj(x)
        out = out.reshape(B, F_dim, T_f, self.n_speakers, 2)
        mask_r = out[..., 0].clamp(-5, 5)
        mask_i = out[..., 1].clamp(-5, 5)
        real_i = spec_real.unsqueeze(-1)
        imag_i = spec_imag.unsqueeze(-1)
        sep_r = mask_r * real_i - mask_i * imag_i
        sep_i = mask_r * imag_i + mask_i * real_i
        outputs = []
        for i in range(self.n_speakers):
            s = torch.complex(sep_r[..., i], sep_i[..., i])
            outputs.append(self.stft.inverse(s, length=T))


In [ ]:
"""
Training script for TF-GridNet speaker separation.

Key training decisions:
  - PIT (Permutation Invariant Training) loss using SI-SNR objective
  - AdamW optimizer with weight decay
  - Warmup + cosine annealing LR schedule (better than ReduceLROnPlateau for this)
  - Gradient clipping at norm 5
  - Mixed precision (fp16) for speed on Kaggle T4/P100
  - Checkpoint every epoch; keep best val loss
"""

import os
import math
import time
import numpy as np
from itertools import permutations

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import 
from model import TFGridNet
from dataset import build_dataloaders


# ─── Loss Functions ──────────────────────────────────────────────────────────

def si_snr(target, estimate):
    """
    Scale-Invariant Signal-to-Noise Ratio.
    target, estimate: [B, T]
    Returns: [B]
    """
    # Zero-mean
    target   = target   - target.mean(dim=-1, keepdim=True)
    estimate = estimate - estimate.mean(dim=-1, keepdim=True)

    # Project estimate onto target
    dot           = (target * estimate).sum(dim=-1, keepdim=True)
    target_energy = (target * target).sum(dim=-1, keepdim=True) + 1e-8
    s_target      = dot * target / target_energy

    e_noise = estimate - s_target

    si_snr_val = 10 * torch.log10(
        (s_target * s_target).sum(dim=-1) /
        ((e_noise * e_noise).sum(dim=-1) + 1e-8)
    )
    return si_snr_val  # [B]


def pit_sisnr_loss(estimates, targets):
    """
    Permutation Invariant Training loss using SI-SNR.
    estimates: [B, n_speakers, 1, T]
    targets:   [B, n_speakers, 1, T]
    Returns scalar loss (negative mean SI-SNR under best permutation).
    """
    estimates = estimates.squeeze(2)  # [B, n_speakers, T]
    targets   = targets.squeeze(2)    # [B, n_speakers, T]

    B, S, T = targets.shape
    perms = list(permutations(range(S)))

    # Compute SI-SNR for all permutations
    # Stack all at once for efficiency
    all_snr = torch.zeros(B, len(perms), device=targets.device)

    for p_idx, perm in enumerate(perms):
        snr_sum = torch.zeros(B, device=targets.device)
        for i, j in enumerate(perm):
            snr_sum += si_snr(targets[:, j, :], estimates[:, i, :])
        all_snr[:, p_idx] = snr_sum / S

    # Best permutation per sample
    best_snr, _ = all_snr.max(dim=1)  # [B]
    return -best_snr.mean()


# ─── Metrics ─────────────────────────────────────────────────────────────────

def si_snri_metric(mixture, estimates, targets):
    """
    SI-SNR improvement: SI-SNR(output) - SI-SNR(mixture as estimate).
    All inputs: [B, n_speakers, 1, T] except mixture: [B, 1, T]
    Returns mean SI-SNRi in dB over best permutation.
    """
    estimates = estimates.squeeze(2)  # [B, S, T]
    targets   = targets.squeeze(2)
    mixture_1d = mixture.squeeze(1)   # [B, T]

    B, S, T = targets.shape
    perms = list(permutations(range(S)))

    batch_sisnri = []
    for b in range(B):
        best = float('-inf')
        for perm in perms:
            snri = 0.0
            for i, j in enumerate(perm):
                snr_out = si_snr(targets[b:b+1, j, :], estimates[b:b+1, i, :]).item()
                snr_mix = si_snr(targets[b:b+1, j, :], mixture_1d[b:b+1, :]).item()
                snri += snr_out - snr_mix
            snri /= S
            if snri > best:
                best = snri
        batch_sisnri.append(best)

    return float(np.mean(batch_sisnri))


# ─── LR Schedule: Warmup + Cosine Annealing ──────────────────────────────────

def get_scheduler(optimizer, warmup_steps, total_steps, min_lr_ratio=0.05):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ─── Training Loop ────────────────────────────────────────────────────────────

def train(config):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    if device.type == 'cuda':

    # Data
    train_loader, val_loader, test_loader = build_dataloaders(
        train_audio_dir  = config['train_audio_dir'],
        librimix_dir     = config.get('librimix_dir', None),
        n_speakers       = config['n_speakers'],
        sample_rate      = config['sample_rate'],
        segment_seconds  = config['segment_seconds'],
        batch_size       = config['batch_size'],
        n_train_samples  = config['n_train_samples'],
        num_workers      = config['num_workers'],
    )

    # Model
    model = TFGridNet(
        n_speakers   = config['n_speakers'],
        sample_rate  = config['sample_rate'],
        n_fft        = config['n_fft'],
        hop_length   = config['hop_length'],
        win_length   = config['win_length'],
        embed_dim    = config['embed_dim'],
        n_blocks     = config['n_blocks'],
        n_heads      = config['n_heads'],
        lstm_hidden  = config['lstm_hidden'],
        cross_heads  = config['cross_heads'],
        dropout      = config['dropout'],
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    

    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr           = config['lr'],
        weight_decay = config['weight_decay'],
        betas        = (0.9, 0.999),
    )

    # Scheduler
    total_steps   = len(train_loader) * config['n_epochs']
    warmup_steps  = len(train_loader) * config['warmup_epochs']
    scheduler     = get_scheduler(optimizer, warmup_steps, total_steps)

    # Mixed precision
    
    # Resume from checkpoint if exists
    start_epoch   = 0
    best_val_loss = float('inf')
    ckpt_path     = config.get('checkpoint_dir', '/kaggle/working/checkpoints')
    os.makedirs(ckpt_path, exist_ok=True)

    resume_path = os.path.join(ckpt_path, 'latest.pt')
    if os.path.exists(resume_path):
        
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
                start_epoch   = ckpt['epoch'] + 1
        best_val_loss = ckpt['best_val_loss']
        

    # ── Main training loop ───────────────────────────────────────────────────
    for epoch in range(start_epoch, config['n_epochs']):
        epoch_start = time.time()

        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        train_losses = []

        for batch_idx, (mixture, sources) in enumerate(train_loader):
            mixture = mixture.to(device, non_blocking=True)   # [B, 1, T]
            sources = sources.to(device, non_blocking=True)   # [B, S, 1, T]

            optimizer.zero_grad(set_to_none=True)

            
                estimated = model(mixture)                    # [B, S, T]
                # Add channel dim to match sources shape
                estimated = estimated.unsqueeze(2)            # [B, S, 1, T]
                loss = pit_sisnr_loss(estimated, sources)

                                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                                    scheduler.step()

            train_losses.append(loss.item())

            if batch_idx % 100 == 0:
                lr = optimizer.param_groups[0]['lr']

        avg_train = np.mean(train_losses)

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        val_losses  = []
        val_sisnris = []

        with torch.no_grad():
            for mixture, sources in val_loader:
                mixture = mixture.to(device, non_blocking=True)
                sources = sources.to(device, non_blocking=True)

                
                    estimated = model(mixture).unsqueeze(2)

                loss = pit_sisnr_loss(estimated, sources)
                val_losses.append(loss.item())

                sisnri = si_snri_metric(mixture, estimated, sources)
                val_sisnris.append(sisnri)

        avg_val      = np.mean(val_losses)
        avg_sisnri   = np.mean(val_sisnris)
        elapsed      = time.time() - epoch_start

        
        
        
        
        
        

        # ── Checkpoint ───────────────────────────────────────────────────────
        state = {
            'epoch':         epoch,
            'model':         model.state_dict(),
            'optimizer':     optimizer.state_dict(),
            'scheduler':     scheduler.state_dict(),
            '            'best_val_loss': best_val_loss,
            'val_sisnri':    avg_sisnri,
        }
        torch.save(state, resume_path)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(state, os.path.join(ckpt_path, 'best.pt'))
            ")

    
    return model


# ─── Evaluation ──────────────────────────────────────────────────────────────

def evaluate(config):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    _, _, test_loader = build_dataloaders(
        train_audio_dir = config['train_audio_dir'],
        librimix_dir    = config.get('librimix_dir', None),
        n_speakers      = config['n_speakers'],
        sample_rate     = config['sample_rate'],
        segment_seconds = config['segment_seconds'],
        batch_size      = config['batch_size'],
        num_workers     = config['num_workers'],
    )

    model = TFGridNet(
        n_speakers  = config['n_speakers'],
        sample_rate = config['sample_rate'],
        n_fft       = config['n_fft'],
        hop_length  = config['hop_length'],
        win_length  = config['win_length'],
        embed_dim   = config['embed_dim'],
        n_blocks    = config['n_blocks'],
        n_heads     = config['n_heads'],
        lstm_hidden = config['lstm_hidden'],
        cross_heads = config['cross_heads'],
        dropout     = 0.0,   # no dropout at test time
    ).to(device)

    ckpt_path = os.path.join(config.get('checkpoint_dir', '/kaggle/working/checkpoints'), 'best.pt')
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f"Loaded best model from epoch {ckpt['epoch']+1}, "
          f"val SI-SNRi: {ckpt['val_sisnri']:.2f} dB")

    all_sisnri = []
    with torch.no_grad():
        for mixture, sources in test_loader:
            mixture = mixture.to(device)
            sources = sources.to(device)
            estimated = model(mixture).unsqueeze(2)
            sisnri = si_snri_metric(mixture, estimated, sources)
            all_sisnri.append(sisnri)

    mean_sisnri = np.mean(all_sisnri)
    std_sisnri  = np.std(all_sisnri)
    
    return mean_sisnri


# ─── Config ──────────────────────────────────────────────────────────────────

CONFIG = {
    # Paths -- update these to your Kaggle paths
    'train_audio_dir':  '/kaggle/input/librispeech-clean/LibriSpeech/train-clean-100',
    'librimix_dir':     None,   # set to LibriMix path if you have it, else None
    'checkpoint_dir':   '/kaggle/working/checkpoints',

    # Task
    'n_speakers':       2,
    'sample_rate':      8000,

    # STFT
    'n_fft':            256,    # → 129 freq bins
    'hop_length':       128,    # 50% overlap
    'win_length':       256,

    # Model
    'embed_dim':        32,     # D per freq bin
    'n_blocks':         4,      # B (paper uses 6; reduce to 4 for faster experiments)
    'n_heads':          4,      # intra-frame attention heads (must divide embed_dim)
    'lstm_hidden':      128,    # H for sub-band LSTM
    'cross_heads':      4,      # cross-frame attention heads (must divide n_freqs*embed_dim = 129*64 = 8256; 4 divides it)
    'dropout':          0.1,

    # Training
    'n_epochs':         100,
    'warmup_epochs':    10,
    'batch_size':       4,      # reduce to 2 if OOM
    'lr':               1e-4,
    'weight_decay':     1e-2,
    'n_train_samples':  8000,
    'segment_seconds':  4.0,
    'num_workers':      2,
}


if __name__ == '__main__':
    import sys
    mode = sys.argv[1] if len(sys.argv) > 1 else 'train'

    if mode == 'train':
        train(CONFIG)
    elif mode == 'eval':
        evaluate(CONFIG)
    else:
        
